# 12 — Final Report Outputs and Export Pack

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("12_final_report_outputs", total_steps=5, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook gathers the final tables, network summaries, visual files, and interpretation-ready outputs into a report asset pack.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
# Load important final tables if they exist
files_to_collect = {
    "senate_national_results": PROCESSED / "senate_national_candidate_results.csv",
    "candidate_online_vs_election": PROCESSED / "candidate_online_vs_election_metrics.csv",
    "candidate_rank_comparison": PROCESSED / "candidate_rank_comparison_online_vs_votes.csv",
    "hashtag_metrics": PROCESSED / "hashtag_node_metrics.csv",
    "candidate_comention_metrics": PROCESSED / "candidate_comention_node_metrics.csv",
    "candidate_hashtag_metrics": PROCESSED / "candidate_hashtag_bipartite_node_metrics.csv",
    "correlations": TABLES / "10_correlation_online_metrics_vs_votes.csv",
    "top12_overlap": TABLES / "10_top12_overlap_precision_recall.csv",
    "regional_top5": TABLES / "regional_top5_candidates.csv",
}

loaded = {}
for name, path in files_to_collect.items():
    if path.exists():
        loaded[name] = pd.read_csv(path)
        print("Loaded", name, loaded[name].shape)
    else:
        print("Missing", name, path)


In [ ]:
progress.step("Purpose")
# Create a consolidated Excel workbook
if loaded:
    workbook_path = REPORT_ASSETS / "philippine_election_network_science_final_tables.xlsx"
    safe_to_excel(loaded, workbook_path)
    print("Saved workbook:", workbook_path)
else:
    print("No tables loaded yet. Run previous notebooks first.")


In [ ]:
progress.step("Purpose")
# Create a report inventory of visual outputs
inventory_rows = []
for folder in [FIGURES, INTERACTIVE, NETWORKS, TABLES, REPORT_ASSETS]:
    for f in sorted(folder.glob("**/*")):
        if f.is_file():
            inventory_rows.append({
                "folder": folder.name,
                "file": f.name,
                "relative_path": str(f.relative_to(ROOT)),
                "size_kb": round(f.stat().st_size / 1024, 1),
            })

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(REPORT_ASSETS / "report_asset_inventory.csv", index=False)
inventory.head(100)


In [ ]:
progress.step("Purpose")
# Build a concise text summary file for report writing.
summary_lines = []
summary_lines.append("Philippine Election 2025 Network Science Project — Output Summary")
summary_lines.append("=" * 72)
summary_lines.append("")
if "senate_national_results" in loaded:
    nat = loaded["senate_national_results"]
    summary_lines.append("Top 12 national Senate winners based on uploaded outcome dataset:")
    for _, row in nat.sort_values("vote_rank").head(12).iterrows():
        summary_lines.append(f"{int(row['vote_rank']):>2}. {row['candidate']} — {int(row['total_votes']):,} votes")
    summary_lines.append("")
if "correlations" in loaded and not loaded["correlations"].empty:
    corr = loaded["correlations"].sort_values("spearman_r_vs_votes", ascending=False)
    summary_lines.append("Online metrics most aligned with vote totals by Spearman correlation:")
    for _, row in corr.head(5).iterrows():
        summary_lines.append(f"- {row['online_metric']}: Spearman r = {row['spearman_r_vs_votes']:.3f}")
    summary_lines.append("")
if "top12_overlap" in loaded and not loaded["top12_overlap"].empty:
    ov = loaded["top12_overlap"].sort_values("top12_overlap_count", ascending=False)
    summary_lines.append("Best top-12 overlap metrics:")
    for _, row in ov.head(5).iterrows():
        summary_lines.append(f"- {row['online_metric']}: {int(row['top12_overlap_count'])}/12 overlap")
    summary_lines.append("")
summary_lines.append("See outputs/figures, outputs/interactive, outputs/networks, and outputs/report_assets for final files.")

summary_path = REPORT_ASSETS / "final_output_summary.txt"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))


## Final note

After all notebooks are run, the project should have static PNG figures, interactive HTML visualizations, GEXF files for Gephi, CSV tables, and one consolidated Excel workbook for reporting.

## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
